# COMP5339 Assignment 1 — Data Acquisition

**Purpose:** download the two reproducible input sources used by the project: NSW EV charging locations and the Australia-wide 2026 SA4 boundaries.

Run this notebook from the **repository root**. It writes inputs to `data/raw/`, which is intentionally Git-ignored. The notebook does not clean, filter, or alter the downloaded source data.


## Source and reproducibility notes

- EV chargers: NSW Government Data portal CKAN API. The notebook resolves the resource URL from the dataset metadata instead of hard-coding a potentially changing resource link.
- SA4 boundaries: Australian Bureau of Statistics (ASGS Edition 4) digital boundary file, GDA2020.
- Download timestamp and SHA-256 checksums are saved to `data/raw/acquisition_manifest.json` for provenance.

Do not commit downloaded data or the manifest to Git; retain them locally or package them only if your submission rules require source data.


In [6]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import zipfile

import requests

def _repo_root() -> Path:
    """Locate the repository root, wherever Jupyter was launched from."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        f"Could not find the repository root (no pyproject.toml above {here})."
    )


PROJECT_ROOT = _repo_root()
RAW_DIR = Path.joinpath(PROJECT_ROOT,"data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

CKAN_DATASET_URL = (
    "https://data.gov.au/data/api/3/action/package_show?"
    "id=69fab867-a077-4e3d-a9ab-c169681ac877"
)
ABS_SA4_URL = (
    "https://www.abs.gov.au/statistics/standards/"
    "australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/"
    "access-and-downloads/digital-boundary-files/"
    "SA4_2026_AUST_SHP_GDA2020.zip"
)

EV_CSV_PATH = RAW_DIR / "ev_charging_locations.csv"
SA4_ZIP_PATH = RAW_DIR / "SA4_2026_AUST_SHP_GDA2020.zip"
SA4_EXTRACT_DIR = RAW_DIR / "sa4_shapefile"
MANIFEST_PATH = RAW_DIR / "acquisition_manifest.json"

print(f"Writing source data under: {RAW_DIR.resolve()}")


Writing source data under: D:\Data Engineering\comp5339-a1\data\raw


## Helper functions

Downloads are streamed to disk, checked for HTTP errors, and represented in the manifest with a checksum. Existing source files are retained unless `force_download=True`, preventing accidental repeated downloads.


In [7]:
def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def download_file(url: str, destination: Path, *, force_download: bool = False) -> dict:
    """Download a file without changing it; return provenance metadata."""
    if destination.exists() and not force_download:
        print(f"Using existing file: {destination}")
    else:
        print(f"Downloading: {url}")
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with destination.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        print(f"Saved: {destination}")

    return {
        "url": url,
        "local_path": str(destination),
        "bytes": destination.stat().st_size,
        "sha256": file_sha256(destination),
    }


def resolve_ev_resource_url() -> str:
    """Locate the named CSV resource from the government CKAN catalogue."""
    response = requests.get(CKAN_DATASET_URL, timeout=60)
    response.raise_for_status()
    resources = response.json()["result"]["resources"]

    matches = [
        resource for resource in resources
        if resource.get("format", "").upper() == "CSV"
        and resource.get("name") == "EV Charging Locations in NSW"
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one EV Charging Locations in NSW CSV resource; found {len(matches)}."
        )
    return matches[0]["url"]


## Acquire EV charger locations

This is the raw input for the cleaning notebook. It is saved with the repository-standard filename `ev_charging_locations.csv`.


In [8]:
FORCE_DOWNLOAD = False  # Set True only when intentionally refreshing a source.

ev_resource_url = resolve_ev_resource_url()
ev_manifest = download_file(ev_resource_url, EV_CSV_PATH, force_download=FORCE_DOWNLOAD)
ev_manifest["dataset_api_url"] = CKAN_DATASET_URL
ev_manifest["resource_name"] = "EV Charging Locations in NSW"

print(json.dumps(ev_manifest, indent=2))


Downloading: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv
Saved: D:\Data Engineering\comp5339-a1\data\raw\ev_charging_locations.csv
{
  "url": "https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv",
  "local_path": "D:\\Data Engineering\\comp5339-a1\\data\\raw\\ev_charging_locations.csv",
  "bytes": 283033,
  "sha256": "43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a",
  "dataset_api_url": "https://data.gov.au/data/api/3/action/package_show?id=69fab867-a077-4e3d-a9ab-c169681ac877",
  "resource_name": "EV Charging Locations in NSW"
}


## Acquire and inspect national SA4 boundaries

The full Australia-wide file is retained. Do **not** pre-filter to NSW: spatial joins must be able to match border-adjacent chargers to a neighbouring state where appropriate. The extracted shapefile is retained under `data/raw/sa4_shapefile/`.


In [9]:
sa4_manifest = download_file(ABS_SA4_URL, SA4_ZIP_PATH, force_download=FORCE_DOWNLOAD)

SA4_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SA4_ZIP_PATH) as archive:
    archive.extractall(SA4_EXTRACT_DIR)

shapefiles = sorted(SA4_EXTRACT_DIR.rglob("*.shp"))
if not shapefiles:
    raise FileNotFoundError("No .shp file was found after extracting the ABS archive.")

sa4_manifest["extraction_directory"] = str(SA4_EXTRACT_DIR)
sa4_manifest["shapefiles"] = [str(path) for path in shapefiles]
print("Extracted shapefile(s):")
for path in shapefiles:
    print(" -", path)


Downloading: https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip
Saved: D:\Data Engineering\comp5339-a1\data\raw\SA4_2026_AUST_SHP_GDA2020.zip
Extracted shapefile(s):
 - D:\Data Engineering\comp5339-a1\data\raw\sa4_shapefile\SA4_2026_AUST_GDA2020.shp


## Save acquisition provenance

The manifest captures exactly what was retrieved in this run. It can be used to explain data lineage without placing raw source data into version control.


In [10]:
manifest = {
    "acquired_at_utc": datetime.now(timezone.utc).isoformat(),
    "ev_chargers": ev_manifest,
    "sa4_boundaries": sa4_manifest,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Wrote provenance manifest: {MANIFEST_PATH}")

# Basic existence checks for the next pipeline stages.
assert EV_CSV_PATH.exists()
assert SA4_ZIP_PATH.exists()
assert shapefiles


Wrote provenance manifest: D:\Data Engineering\comp5339-a1\data\raw\acquisition_manifest.json


## Handoff to the team pipeline

1. Run this notebook once to populate `data/raw/`.
2. Run `notebooks/integration_quality_B.ipynb` to produce cleaning and quality outputs in `data/interim/`.
3. Run the spatial integration workflow against the unfiltered national SA4 shapefile.

Commit this notebook and the implementation under `src/acquisition/`, but not the downloaded `data/raw/` files.
